# E02 seed42: pinned Colab launcher

Recorded source and commands for the completed 2026-10-12 pilot. This notebook calls the shared package and contains no model implementation. Select a T4 runtime. A fresh run requires explicit authorization and newly verified Drive quota; the default controls prevent starting another research run. Checkpoint exports occur at every 250-update validation boundary.


In [ ]:
RESEARCH_AUTHORIZED = False
DRIVE_AVAILABLE_BYTES = None  # Read current free quota in Google Drive Storage before setting.


In [ ]:
import sys, platform, subprocess, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
assert Path('/content/drive/MyDrive').is_dir()
print('Drive mounted and MyDrive visible')

In [ ]:
from pathlib import Path
import hashlib, json, shutil, uuid
drive_root = Path('/content/drive/MyDrive/Nano-Language-Model/E02_seed42')
drive_root.mkdir(parents=True, exist_ok=True)
pilot_store = drive_root / ('pilot_' + uuid.uuid4().hex)
pilot_store.mkdir(exist_ok=False)
probe = pilot_store / 'write_probe.txt'
probe.write_text('E02 durable storage preflight\n', encoding='utf-8')
expected = hashlib.sha256(probe.read_bytes()).hexdigest()
assert hashlib.sha256(probe.read_bytes()).hexdigest() == expected
usage = shutil.disk_usage(pilot_store)
print(json.dumps({'pilot_store': str(pilot_store), 'probe_sha256': expected, 'free_bytes': usage.free, 'total_bytes': usage.total}, indent=2))

In [ ]:
from pathlib import Path
import subprocess, json, hashlib
ROOT = Path('/content/e02_source')
COMMIT = '2cf39518492652e2ed05e459318e6166e4275814'
assert not ROOT.exists()
subprocess.run(['git','clone','--quiet','https://github.com/rayyan-merchant/Nano-Language-Model.git',str(ROOT)],check=True)
subprocess.run(['git','checkout','--quiet','--detach',COMMIT],cwd=ROOT,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip() == COMMIT
assert not subprocess.check_output(['git','status','--porcelain'],cwd=ROOT,text=True).strip()
source = {p.relative_to(ROOT).as_posix():p.read_text(encoding='utf-8') for folder in ('src','configs','experiments/configs') for p in (ROOT/folder).rglob('*') if p.suffix in {'.py','.json'}}
code_hash = hashlib.sha256(json.dumps(source,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
assert code_hash == 'cdea42adb930904fb0cefcd282d784bb3ac8caa311d3f1e12ef568622f9c12d6'
print('Clean pinned commit:', COMMIT, 'Core hash:', code_hash)

In [ ]:
import os, sys, subprocess
from pathlib import Path
SETUP_ENV = dict(os.environ, UV_PYTHON_INSTALL_DIR='/content/e02_python', UV_CACHE_DIR='/content/e02_uv_cache', UV_NATIVE_TLS='true', CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='1', MPLCONFIGDIR='/content/e02_mpl', MPLBACKEND='Agg')
subprocess.run([sys.executable,'-m','pip','install','--target','/content/e02_tools','uv==0.12.7'],env=SETUP_ENV,check=True)
UV=Path('/content/e02_tools/bin/uv')
assert UV.is_file()
subprocess.run([str(UV),'python','install','3.11.14'],env=SETUP_ENV,check=True)
subprocess.run([str(UV),'venv','--seed','--python','3.11.14','/content/e02_venv'],env=SETUP_ENV,check=True)
print('Python 3.11.14 environment created')

In [ ]:
PY=Path('/content/e02_venv/bin/python')
freeze=(ROOT/'docs/qualifications/evidence/rayyan_requirements_freeze.txt').read_text().splitlines()
lock=Path('/content/e02_qualified_lock.txt')
lock.write_text('\n'.join(x for x in freeze if not x.startswith('-e '))+'\n')
print('Installing CUDA PyTorch and exact E01 dependency freeze...',flush=True)
r=subprocess.run([str(UV),'pip','install','--python',str(PY),'torch==2.5.1+cu124','--index-url','https://download.pytorch.org/whl/cu124'],env=SETUP_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print('torch install status',r.returncode,r.stdout[-1200:],flush=True)
r.check_returncode()
r=subprocess.run([str(UV),'pip','install','--python',str(PY),'-r',str(lock)],env=SETUP_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print('freeze install status',r.returncode,r.stdout[-1200:],flush=True)
r.check_returncode()
r=subprocess.run([str(UV),'pip','install','--python',str(PY),'-e',str(ROOT),'--no-deps'],env=SETUP_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print('package install status',r.returncode,r.stdout[-1000:],flush=True)
r.check_returncode()
subprocess.run([str(PY),'-m','pip','check'],env=SETUP_ENV,check=True)
print('Pinned environment installed',flush=True)

In [ ]:
gpu_code="import torch; assert torch.__version__=='2.5.1+cu124'; assert torch.version.cuda=='12.4'; assert torch.cuda.is_available(); x=torch.randn(256,256,device='cuda',requires_grad=True); y=(x@x).square().mean(); y.backward(); assert torch.isfinite(y) and torch.isfinite(x.grad).all(); print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0),float(y))"
r=subprocess.run([str(PY),'-c',gpu_code],env=SETUP_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print('CUDA numerical check:',r.returncode,r.stdout,flush=True)
r.check_returncode()
installed=subprocess.check_output([str(PY),'-m','pip','freeze'],env=SETUP_ENV,text=True).splitlines()
normalize=lambda lines: sorted(x for x in lines if x and not x.startswith('-e '))
assert normalize(installed)==normalize(freeze), 'Dependency freeze mismatch'
print('Exact dependency freeze verified:',len(normalize(installed)),'packages')
assert not subprocess.check_output(['git','status','--porcelain'],cwd=ROOT,text=True).strip(), 'Checkout became dirty'
r=subprocess.run([str(PY),'-m','pytest','-q'],cwd=ROOT,env=SETUP_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print('Automated tests:',r.returncode,r.stdout[-3000:],flush=True)
r.check_returncode()

In [ ]:
DATA=Path('/content/e02_data')
assert not DATA.exists()
r=subprocess.run([str(PY),'-m','nano_lm.prepare','--output',str(DATA)],cwd=ROOT,env=SETUP_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print('Dataset preparation status:',r.returncode,r.stdout[-1500:],flush=True)
r.check_returncode()
r=subprocess.run([str(PY),'-m','nano_lm.prepare','--output',str(DATA),'--validate'],cwd=ROOT,env=SETUP_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print('Independent manifest validation status:',r.returncode,flush=True)
r.check_returncode()
manifest=json.loads((DATA/'manifest.json').read_text())
reference=json.loads((ROOT/'docs/evidence/dataset_manifest.json').read_text())
assert all(manifest[k]==reference[k] for k in ('source','split','partitions','tokenizers','manifest_hash'))
assert manifest['source']['sha256']=='86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed'
assert manifest['tokenizers']['bpe']['hash']=='2d7b1e84c8055eb5b9483a3fe6e4e3c9e348f5a28506ba19a0dc1644f0b36d55'
print('Pinned BPE counts:',manifest['tokenizers']['bpe']['tokens'])
print('Partition bytes:',{k:v['bytes'] for k,v in manifest['partitions'].items()})

In [ ]:
SMOKE_RUNS=Path('/content/e02_smoke_runs')
SMOKE_EXPORTS=pilot_store/'smoke_exports'
smoke_cmd=[str(PY),'-m','nano_lm.train','--root',str(ROOT),'--dataset',str(DATA),'--experiment','E02','--seed','42','--steps','2','--device','cuda','--precision','float32','--run-kind','smoke','--tokens-per-update','128','--micro-batch-size','1','--eval-every','1','--checkpoint-every','1','--eval-max-bytes','512','--output',str(SMOKE_RUNS),'--export-root',str(SMOKE_EXPORTS)]
r=subprocess.run(smoke_cmd,cwd=ROOT,env=SETUP_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print('E02 CUDA smoke and Drive export:',r.returncode,r.stdout[-3000:],flush=True)
r.check_returncode()
print('Smoke exports:',[p.name for p in SMOKE_EXPORTS.iterdir()])

In [ ]:
SNAPSHOTS=sorted(p for p in SMOKE_EXPORTS.iterdir() if p.is_dir())
assert len(SNAPSHOTS)>=2
checks=[subprocess.run([str(PY),'-m','nano_lm.storage','verify',str(p)],cwd=ROOT,env=SETUP_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) for p in SNAPSHOTS]
assert all(r.returncode==0 for r in checks), [r.stdout[-1000:] for r in checks]
SNAPSHOT_MANIFESTS={p:json.loads((p/'export_manifest.json').read_text()) for p in SNAPSHOTS}
LATEST=max(SNAPSHOTS,key=lambda p:(SNAPSHOT_MANIFESTS[p]['step'],len(SNAPSHOT_MANIFESTS[p]['files'])))
assert SNAPSHOT_MANIFESTS[LATEST]['step']==2
RESTORE_ROOT=Path('/content/e02_smoke_restore')
assert not RESTORE_ROOT.exists()
r=subprocess.run([str(PY),'-m','nano_lm.storage','restore',str(LATEST),str(RESTORE_ROOT)],cwd=ROOT,env=SETUP_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print('Fresh restore:',r.returncode,r.stdout[-1000:],flush=True)
r.check_returncode()
RECOVERED=RESTORE_ROOT/SNAPSHOT_MANIFESTS[LATEST]['run_id']/'checkpoints/step_000002.pt'
assert RECOVERED.is_file()
r=subprocess.run([str(PY),'-m','nano_lm.generate','--checkpoint',str(RECOVERED),'--prompt','To be','--max-new-tokens','8','--greedy'],cwd=ROOT,env=SETUP_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print('Recovered checkpoint inference:',r.returncode,r.stdout[-1000:],flush=True)
r.check_returncode()
print('Verified',len(SNAPSHOTS),'Drive snapshot manifests and file hashes')

In [ ]:
SMOKE_RESUME_EXPORTS=pilot_store/'smoke_resume_exports'
resume_cmd=smoke_cmd.copy()
resume_cmd[resume_cmd.index('--steps')+1]='4'
resume_cmd[resume_cmd.index('--export-root')+1]=str(SMOKE_RESUME_EXPORTS)
resume_cmd.extend(['--resume',str(RECOVERED)])
r=subprocess.run(resume_cmd,cwd=ROOT,env=SETUP_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print('Recovered E02 CUDA continuation:',r.returncode,r.stdout[-2600:],flush=True)
r.check_returncode()
assert (RECOVERED.parent/'step_000004.pt').is_file()
print('Recovered run reached smoke step 4; research step count remains zero')

In [ ]:
audit_code="import json; from pathlib import Path; from nano_lm.config import load_experiment; from nano_lm.model import GPT; from nano_lm.train import TrainSettings; from nano_lm.bpe import from_artifact; r=Path('/content/e02_source'); d=Path('/content/e02_data'); tok=from_artifact(json.loads((d/'bpe.json').read_text())); cfg,base,var=load_experiment(r,'E02',1024); s=TrainSettings(seed=42,device='cuda',precision='float32',run_kind='research',eval_every=250,checkpoint_every=500); s.validate(cfg.context_length); assert GPT(cfg).parameter_count()==2173056; assert s.tokens_per_update==4096 and s.micro_batch_size==4 and s.tokens_per_update//(cfg.context_length*s.micro_batch_size)==8; assert var['tokenizer']=='bpe' and var['model_overrides']=={}; assert base['optimizer']=='AdamW' and base['scheduler']=='constant'; assert all(tok.decode(tok.encode((d/(part+'.txt')).read_text()))==(d/(part+'.txt')).read_text() for part in ('train','validation','test')); print('E02 config, 2173056 parameters, accumulation 8, BPE roundtrips PASS')"
r=subprocess.run([str(PY),'-c',audit_code],cwd=ROOT,env=SETUP_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print('Canonical E02 audit:',r.returncode,r.stdout[-1200:],flush=True)
r.check_returncode()
SMOKE_CKPT=next(SMOKE_RUNS.rglob('step_000002.pt'))
ckpt_bytes=SMOKE_CKPT.stat().st_size
conservative_bytes=ckpt_bytes*100
assert DRIVE_AVAILABLE_BYTES is not None and conservative_bytes < DRIVE_AVAILABLE_BYTES
preflight={'source_commit':COMMIT,'core_hash':code_hash,'python':'3.11.14','torch':'2.5.1+cu124','cuda':'12.4','gpu':'Tesla T4','dependency_freeze_matches_E01':True,'automated_tests_passed':35,'raw_sha256':manifest['source']['sha256'],'bpe_sha256':manifest['tokenizers']['bpe']['hash'],'storage':str(pilot_store),'verified_drive_available_bytes':DRIVE_AVAILABLE_BYTES,'smoke_checkpoint_bytes':ckpt_bytes,'conservative_100_checkpoint_bytes':conservative_bytes,'verified_smoke_snapshots':len(SNAPSHOTS),'recovered_inference':True,'recovered_cuda_resume_to_smoke_step':4,'research_optimizer_updates':0}
(pilot_store/'preflight.json').write_text(json.dumps(preflight,indent=2))
print(json.dumps(preflight,indent=2))

In [ ]:
assert RESEARCH_AUTHORIZED, 'Separate explicit research-run authorization is required'
from datetime import datetime, timezone
RESEARCH_RUNS=Path('/content/e02_research_runs')
RESEARCH_EXPORTS=pilot_store/'research_snapshots'
assert not RESEARCH_RUNS.exists() and not RESEARCH_EXPORTS.exists()
train_cmd=[str(PY),'-m','nano_lm.train','--root',str(ROOT),'--dataset',str(DATA),'--experiment','E02','--seed','42','--steps','5000','--device','cuda','--precision','float32','--run-kind','research','--tokens-per-update','4096','--micro-batch-size','4','--optimizer','AdamW','--learning-rate','0.0003','--weight-decay','0.01','--gradient-clip','1.0','--eval-every','250','--checkpoint-every','500','--conditioning-bytes','256','--eval-max-bytes','0','--eval-mode','sliding','--eval-batch-size','32','--output',str(RESEARCH_RUNS),'--export-root',str(RESEARCH_EXPORTS)]
launch={'source_commit':COMMIT,'core_hash':code_hash,'command':train_cmd,'launched_utc':datetime.now(timezone.utc).isoformat(),'authorized_max_updates':5000,'experiment':'E02','seed':42}
(pilot_store/'research_launch.json').write_text(json.dumps(launch,indent=2))
research_log=(pilot_store/'research_train.log').open('w')
research_proc=subprocess.Popen(train_cmd,cwd=ROOT,env=SETUP_ENV,stdout=research_log,stderr=subprocess.STDOUT,start_new_session=True)
print('E02 research process PID:',research_proc.pid,'running:',research_proc.poll() is None)
assert research_proc.poll() is None

In [ ]:
runs=list(RESEARCH_RUNS.iterdir()) if RESEARCH_RUNS.exists() else []
run_dir=runs[0] if len(runs)==1 else None
lines=(run_dir/'events.jsonl').read_text().splitlines() if run_dir and (run_dir/'events.jsonl').exists() else []
last=json.loads(lines[-1]) if lines else {}
validations=[json.loads(x)['metric'] for x in lines if json.loads(x).get('event')=='step' and 'validation' in json.loads(x).get('metric',{})]
snapshot_count=len(list(RESEARCH_EXPORTS.iterdir())) if RESEARCH_EXPORTS.exists() else 0
print(json.dumps({'pid':research_proc.pid,'returncode':research_proc.poll(),'run_dir':str(run_dir),'last_event':last.get('event'),'last_step':last.get('metric',{}).get('step',last.get('step')),'full_validations':len(validations),'latest_validation_step':validations[-1]['step'] if validations else None,'durable_snapshots':snapshot_count,'log_tail':(pilot_store/'research_train.log').read_text()[-800:]},indent=2))